# Modelado del Efecto Mpemba en Sistemas Coloidales
**Paper de referencia:** Kumar & Bechhoefer, *Nature* **584**, 64 (2020)

Este notebook reproduce y extiende el análisis de las curvas $D_{L1}(t)$ generadas
por la simulación Langevin (`mpemba-x-v2 / modules/langevin`). Se aplican tres
arquitecturas de ML complementarias:

| Modelo | Pregunta que responde |
|--------|----------------------|
| **Neural ODE** (RK4) | ¿Qué dinámica continua $dD/dt = f_\\theta(D)$ subyace a las curvas? |
| **PINN** | ¿Puede una red respetar $dD/dt \\le 0$ mientras ajusta los datos? |
| **PySR** | ¿Qué forma analítica tiene $\\dot{D}(D)$ según la regresión simbólica? |

> La LSTM del borrador original se **reemplaza** por la Neural ODE con integrador RK4
> (error local $O(\\Delta t^5)$ vs. $O(\\Delta t^2)$ de Euler, crítico en la caída rápida inicial).


## 1. Carga de datos y exploración inicial

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.optimize import curve_fit

# ─── Datos ───────────────────────────────────────────────────────────────────
df = pd.read_csv('distances.csv')

t_np  = df['t'].values.astype(np.float32)
Dh_np = df['D_L1_h'].values.astype(np.float32)   # hot
Dw_np = df['D_L1_w'].values.astype(np.float32)   # warm
Dc_np = df['D_L1_c'].values.astype(np.float32)   # cold (control)

t  = torch.tensor(t_np).view(-1, 1)                        # (201, 1)
y  = torch.tensor(np.stack([Dh_np, Dw_np], axis=1))        # (201, 2)

# ─── Detección del cruce (efecto Mpemba) ─────────────────────────────────────
diff      = Dh_np - Dw_np
cross_idx = np.where(np.diff(np.sign(diff)))[0]
t_cross   = t_np[cross_idx] if len(cross_idx) else np.array([])

print(f'Puntos temporales:  {len(t_np)}')
print(f'Rango de t:         {t_np[0]:.4f} → {t_np[-1]:.4f} s  |  Δt = {t_np[1]-t_np[0]:.4f} s')
print(f'D_h(0) = {Dh_np[0]:.4f}   D_w(0) = {Dw_np[0]:.4f}')
print(f'D_inf (hot)  ≈ {Dh_np[-20:].mean():.5f} ± {Dh_np[-20:].std():.5f}')
print(f'D_inf (warm) ≈ {Dw_np[-20:].mean():.5f} ± {Dw_np[-20:].std():.5f}')
if len(t_cross):
    print(f'Cruce(s) detectados en t* ≈ {t_cross} s  → Efecto Mpemba ✓')
else:
    print('Sin cruce detectado')


### Vista rápida de las curvas originales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, logscale in zip(axes, [False, True]):
    ax.plot(t_np, Dh_np, 'r-',  lw=1.5, label=r'$D_{L1}^{hot}$')
    ax.plot(t_np, Dw_np, 'b-',  lw=1.5, label=r'$D_{L1}^{warm}$')
    ax.plot(t_np, Dc_np, 'k:',  lw=1,   label=r'$D_{L1}^{cold}$ (control)')
    for tc in t_cross:
        ax.axvline(tc, color='gray', ls='--', alpha=0.7,
                   label=f'$t^* \\approx {tc:.3f}$ s')
    ax.set_xlabel('t (s)')
    ax.set_ylabel(r'$D_{L1}$')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if logscale:
        ax.set_yscale('log')
        ax.set_xlim(0, 0.15)
        ax.set_title('Escala log — régimen transitorio')
    else:
        ax.set_title('Escala lineal')

plt.suptitle('Curvas $D_{L1}(t)$ — Simulación Langevin (Kumar & Bechhoefer 2020)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


## 2. Neural ODE con integrador RK4

Aprendemos $f_\\theta$ tal que $\\dot{D} = f_\\theta(D)$, integrando con **Runge-Kutta de orden 4**
en lugar de Euler. El error local pasa de $O(\\Delta t^2)$ a $O(\\Delta t^5)$, relevante
en la zona de caída rápida ($t < 0.05$ s) donde $dD_h/dt \\approx -36$ en las
unidades del CSV.

A tiempos largos la expansión en autofunciones (ec. 1 del paper) predice
$D(t) \\approx |a_2|Ve^{-\\lambda_2 t}$, es decir $\\dot{D} \\approx -\\lambda_2 D$.
Si $f_\\theta$ converge a esa forma, la red está redescubriendo el eigenvalor dominante.


In [ ]:
# ─── Arquitectura ────────────────────────────────────────────────────────────
class ODEFunc(nn.Module):
    """Aproxima dD/dt = f_theta(D).  Entrada/salida: (2,)"""
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
    def forward(self, state):
        return self.net(state)

# ─── Integrador RK4 ──────────────────────────────────────────────────────────
def rk4_integrate(func, y0, t_vec):
    """Integra y'=func(y) desde y0 usando RK4 paso a paso.
    t_vec : tensor (N,1)  |  Retorna tensor (N, 2).
    """
    t_flat = t_vec.squeeze()   # (N,)
    ys = [y0]
    y_cur = y0
    for i in range(1, len(t_flat)):
        dt = (t_flat[i] - t_flat[i-1]).item()
        k1 = func(y_cur)
        k2 = func(y_cur + 0.5 * dt * k1)
        k3 = func(y_cur + 0.5 * dt * k2)
        k4 = func(y_cur + dt * k3)
        y_cur = y_cur + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        ys.append(y_cur)
    return torch.stack(ys)     # (N, 2)

# ─── Entrenamiento ───────────────────────────────────────────────────────────
torch.manual_seed(42)
ode_func      = ODEFunc(hidden=32)
optimizer_ode = torch.optim.Adam(ode_func.parameters(), lr=5e-3)
scheduler_ode = torch.optim.lr_scheduler.StepLR(optimizer_ode, step_size=300, gamma=0.5)
criterion     = nn.MSELoss()

history_ode = []
for epoch in range(1200):
    optimizer_ode.zero_grad()
    y_pred_ode = rk4_integrate(ode_func, y[0], t)
    loss = criterion(y_pred_ode, y)
    loss.backward()
    optimizer_ode.step()
    scheduler_ode.step()
    history_ode.append(loss.item())
    if epoch % 300 == 0:
        print(f'Epoch {epoch:4d} | Loss: {loss.item():.6f}')

print(f'Epoch 1199 | Loss: {history_ode[-1]:.6f}')


## 3. PINN con restricción termodinámica

La pérdida total combina un término de datos y uno físico:

$$\\mathcal{L} = \\lambda_{\\text{data}}\\,\\mathcal{L}_{\\text{MSE}}
+ \\lambda_{\\text{phys}}\\,\\mathcal{L}_{\\text{phys}}$$

donde la restricción activa penaliza violaciones de monotonicidad:

$$\\mathcal{L}_{\\text{phys}} = \\frac{1}{N}\\sum_i
\\bigl[\\max(0, \\dot{D}_h)\\bigr]^2 + \\bigl[\\max(0, \\dot{D}_w)\\bigr]^2$$

Esta condición ($dD/dt \\le 0$) es exactamente la que Lu & Raz (PNAS 2017) exigen
a toda medida de distancia válida para definir el efecto Mpemba.

> **Para experimentar:** si PySR confirma $\\dot{D} \\approx -k D$, sustituye
> `loss_phys` por el residuo de la ODE lineal `dD/dt + k*D = 0`.
> Ajusta `lambda_phys` para controlar el balance datos/física.


In [ ]:
# ─── Arquitectura ────────────────────────────────────────────────────────────
class PINN(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),      nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
    def forward(self, t_in):   # t_in: (N, 1)  →  (N, 2)
        return self.net(t_in)

# ─── Función de pérdida editable ─────────────────────────────────────────────
def pinn_loss(model, t_data, y_data, t_phys,
              lambda_data=1.0, lambda_phys=0.5):
    """
    PARA EDITAR LA RESTRICCIÓN FÍSICA:
    ─────────────────────────────────
    Opción A (actual): monotonicidad pura   dD/dt <= 0
    Opción B: decaimiento exponencial       dD/dt + k*D = 0
              → sustituye loss_phys por:
                k = 5.0  # valor aproximado de PySR / ajuste exponencial
                res_h = dD_dt_h + k * y_p[:, 0:1]
                res_w = dD_dt_w + k * y_p[:, 1:2]
                loss_phys = (res_h**2).mean() + (res_w**2).mean()
    """
    # Data loss
    y_pred = model(t_data)
    l_data = nn.MSELoss()(y_pred, y_data)

    # Physics loss via autograd
    t_p = t_phys.clone().requires_grad_(True)
    y_p = model(t_p)

    dD_dt_h = torch.autograd.grad(
        y_p[:, 0].sum(), t_p, create_graph=True, retain_graph=True)[0]
    dD_dt_w = torch.autograd.grad(
        y_p[:, 1].sum(), t_p, create_graph=True, retain_graph=True)[0]

    # Opción A: penalizar gradiente positivo
    loss_phys = (torch.relu(dD_dt_h)**2).mean() \
              + (torch.relu(dD_dt_w)**2).mean()

    total = lambda_data * l_data + lambda_phys * loss_phys
    return total, l_data, loss_phys

# ─── Entrenamiento ───────────────────────────────────────────────────────────
torch.manual_seed(42)
pinn           = PINN(hidden=64)
optimizer_pinn = torch.optim.Adam(pinn.parameters(), lr=1e-3)
scheduler_pinn = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_pinn, T_max=2000, eta_min=1e-5)

t_phys       = t.clone()
history_pinn = []

for epoch in range(2000):
    optimizer_pinn.zero_grad()
    loss, l_d, l_p = pinn_loss(pinn, t, y, t_phys)
    loss.backward()
    optimizer_pinn.step()
    scheduler_pinn.step()
    history_pinn.append((loss.item(), l_d.item(), l_p.item()))
    if epoch % 500 == 0:
        print(f'Epoch {epoch:4d} | Total: {loss.item():.5f} '
              f'| Data: {l_d.item():.5f}  | Phys: {l_p.item():.5f}')

print(f'Epoch 1999 | Total: {history_pinn[-1][0]:.5f} '
      f'| Data: {history_pinn[-1][1]:.5f}  | Phys: {history_pinn[-1][2]:.5f}')


## 4. Regresión simbólica con PySR

Buscamos $\\dot{D} = f(D)$ en lugar de $D(t)$, porque el paper predice
$\\dot{D} \\approx -\\lambda_2 D$ a tiempos largos (eigenvalor dominante de
Fokker-Planck). Si PySR recupera una relación lineal, $k \\approx \\lambda_2$.

Usamos la **región de relajación lenta** ($t > 0.05$ s) para aislar el
régimen asintótico y evitar el transitorio rápido inicial.


In [ ]:
# !pip install pysr   # descomentar si no está instalado
from pysr import PySRRegressor

# ─── Región asintótica (t > 0.05 s) ──────────────────────────────────────────
mask_slow  = t_np > 0.05
D_h_slow   = Dh_np[mask_slow]

# Derivada numérica (diferencias centradas con numpy)
dDh_dt_np  = np.gradient(Dh_np, t_np)
dDh_slow   = dDh_dt_np[mask_slow]

print(f'Puntos para SR (t > 0.05 s): {mask_slow.sum()}')
print(f'Rango D_h:     [{D_h_slow.min():.4f}, {D_h_slow.max():.4f}]')
print(f'Rango dD_h/dt: [{dDh_slow.min():.4f}, {dDh_slow.max():.4f}]')

# ─── Modelo SR: dD/dt = f(D) ─────────────────────────────────────────────────
X_sr = D_h_slow.reshape(-1, 1)   # feature: D
y_sr = dDh_slow                   # target:  dD/dt

model_sr = PySRRegressor(
    niterations      = 60,
    binary_operators = ["+", "*", "-", "/"],
    unary_operators  = ["exp", "log", "neg"],
    model_selection  = "best",
    elementwise_loss = "loss(prediction, target) = (prediction - target)^2",
    verbosity        = 0,
)
model_sr.fit(X_sr, y_sr)

print("\nTabla de Pareto (complejidad vs. pérdida):")
print(model_sr)
print("\nMejor ecuación simbólica para dD_h/dt = f(D_h):")
print(model_sr.sympy())


## 5. Comparación de modelos y detección del cruce

Panel principal: ajuste de las curvas $D_{L1}(t)$ con cruce marcado.
Paneles secundarios: historia de pérdida, zoom en $t^*$, y relación $\\dot{D}$ vs $D$.


In [ ]:
ode_func.eval()
pinn.eval()

with torch.no_grad():
    y_pred_ode  = rk4_integrate(ode_func, y[0], t).numpy()   # (201, 2)
    y_pred_pinn = pinn(t).numpy()                             # (201, 2)

# ─── Métricas ────────────────────────────────────────────────────────────────
mse_ode_h  = np.mean((y_pred_ode[:,0]  - Dh_np)**2)
mse_ode_w  = np.mean((y_pred_ode[:,1]  - Dw_np)**2)
mse_pinn_h = np.mean((y_pred_pinn[:,0] - Dh_np)**2)
mse_pinn_w = np.mean((y_pred_pinn[:,1] - Dw_np)**2)

print("=== MSE por curva ===")
print(f"Neural ODE  — hot: {mse_ode_h:.6f}   warm: {mse_ode_w:.6f}")
print(f"PINN        — hot: {mse_pinn_h:.6f}   warm: {mse_pinn_w:.6f}")

# ─── Detección cruce en predicciones ─────────────────────────────────────────
def find_cross(arr1, arr2, t_vec):
    d   = arr1 - arr2
    idx = np.where(np.diff(np.sign(d)))[0]
    return t_vec[idx] if len(idx) else np.array([])

t_cross_ode  = find_cross(y_pred_ode[:,0],  y_pred_ode[:,1],  t_np)
t_cross_pinn = find_cross(y_pred_pinn[:,0], y_pred_pinn[:,1], t_np)

print(f"\nCruce en datos originales:  t* = {t_cross} s")
print(f"Cruce Neural ODE:           t* = {t_cross_ode} s")
print(f"Cruce PINN:                 t* = {t_cross_pinn} s")

# ─── Figura ──────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

ax_main = fig.add_subplot(gs[0, :])
ax_loss = fig.add_subplot(gs[1, 0])
ax_zoom = fig.add_subplot(gs[1, 1])
ax_sr   = fig.add_subplot(gs[1, 2])

# Panel 1 — curvas completas
ax = ax_main
ax.plot(t_np, Dh_np,         'r--', lw=1.8, alpha=0.55, label=r'$D_{L1}^{hot}$ (sim)')
ax.plot(t_np, Dw_np,         'b--', lw=1.8, alpha=0.55, label=r'$D_{L1}^{warm}$ (sim)')
ax.plot(t_np, y_pred_ode[:,0], 'r-', lw=1.5,  label='Neural ODE — hot')
ax.plot(t_np, y_pred_ode[:,1], 'b-', lw=1.5,  label='Neural ODE — warm')
ax.plot(t_np, y_pred_pinn[:,0],'r:', lw=2.0,  label='PINN — hot')
ax.plot(t_np, y_pred_pinn[:,1],'b:', lw=2.0,  label='PINN — warm')
for tc in t_cross:
    ax.axvline(tc, color='gray', ls='--', lw=1.2, alpha=0.8,
               label=f'$t^*={tc:.3f}$ s (datos)')
ax.set_xlabel('t (s)'); ax.set_ylabel(r'$D_{L1}$')
ax.set_title('Ajuste de curvas de relajación — Efecto Mpemba ($t_h < t_w$)')
ax.legend(fontsize=8, ncol=3); ax.grid(True, alpha=0.3)

# Panel 2 — historia de pérdida
ax = ax_loss
ax.semilogy(history_ode,               'g-',  lw=1.2, label='Neural ODE')
ax.semilogy([h[0] for h in history_pinn],'m-', lw=1.2, label='PINN total')
ax.semilogy([h[1] for h in history_pinn],'m--',lw=0.9, alpha=0.6, label='PINN data')
ax.semilogy([h[2] for h in history_pinn],'m:', lw=0.9, alpha=0.6, label='PINN phys')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Historia de entrenamiento')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 3 — zoom en zona de cruce
ax = ax_zoom
mask_z = (t_np >= 0.02) & (t_np <= 0.08)
ax.plot(t_np[mask_z], Dh_np[mask_z],          'r--', lw=2,   alpha=0.5, label='Data hot')
ax.plot(t_np[mask_z], Dw_np[mask_z],          'b--', lw=2,   alpha=0.5, label='Data warm')
ax.plot(t_np[mask_z], y_pred_ode[mask_z,0],   'r-',  lw=1.5, label='ODE hot')
ax.plot(t_np[mask_z], y_pred_ode[mask_z,1],   'b-',  lw=1.5, label='ODE warm')
ax.plot(t_np[mask_z], y_pred_pinn[mask_z,0],  'r:',  lw=2.0, label='PINN hot')
ax.plot(t_np[mask_z], y_pred_pinn[mask_z,1],  'b:',  lw=2.0, label='PINN warm')
for tc in t_cross:
    ax.axvline(tc, color='gray', ls='--', lw=1)
ax.set_xlabel('t (s)'); ax.set_ylabel(r'$D_{L1}$')
ax.set_title(r'Zoom — zona de cruce ($t^*$)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Panel 4 — dD/dt vs D (SR + datos)
ax = ax_sr
ax.scatter(D_h_slow, dDh_slow, s=8, alpha=0.5, color='tomato',
           label=r'$dD_h/dt$ numérico ($t>0.05$ s)')
try:
    ax.plot(np.sort(D_h_slow),
            model_sr.predict(np.sort(D_h_slow).reshape(-1,1)),
            'k-', lw=1.8, label='PySR')
except Exception:
    pass
ax.set_xlabel(r'$D_{L1}$'); ax.set_ylabel(r'$dD_{L1}/dt$')
ax.set_title(r'Relación $\dot{D}$ vs $D$ (régimen lento)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de modelos — Kumar & Bechhoefer (2020)', fontsize=12)
plt.savefig('mpemba_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada como mpemba_comparison.png')


## 6. Extracción de $\\lambda_2$ y comparación con la Neural ODE

A tiempos largos la teoría predice (ec. 27 del paper):

$$D(t) \\approx |a_2|\\,V\\,e^{-\\lambda_2 t} + D_\\infty$$

Ajustamos esta forma a los datos para estimar $\\lambda_2$. Si PySR
recuperó $\\dot{D} \\approx -k D$, debe cumplirse $k \\approx \\lambda_2$.


In [ ]:
def exp_decay(t, A, lam, D_inf):
    return A * np.exp(-lam * t) + D_inf

mask_fit = t_np > 0.05   # régimen asintótico

lambdas = {}
for label, D_arr in [('hot', Dh_np), ('warm', Dw_np)]:
    try:
        p0 = [D_arr[mask_fit][0] - D_arr[-10:].mean(),
              5.0,
              D_arr[-10:].mean()]
        popt, pcov = curve_fit(exp_decay, t_np[mask_fit], D_arr[mask_fit],
                               p0=p0, maxfev=8000)
        A, lam, D_inf = popt
        perr = np.sqrt(np.diag(pcov))
        lambdas[label] = lam
        print(f'[{label:4s}]  A  = {A:.5f} ± {perr[0]:.5f}')
        print(f'        λ₂ = {lam:.4f} ± {perr[1]:.4f}  '
              f'→  τ = 1/λ₂ = {1/lam:.4f} s')
        print(f'        D∞ = {D_inf:.5f} ± {perr[2]:.5f}')
        print()
    except RuntimeError as e:
        print(f'[{label}] Ajuste no convergió: {e}')

# ─── ¿Qué aprendió la Neural ODE en ese régimen? ─────────────────────────────
D_mid_h = float(Dh_np[mask_fit].mean())
D_mid_w = float(Dw_np[mask_fit].mean())
D_tensor = torch.tensor([D_mid_h, D_mid_w])

with torch.no_grad():
    dD_pred = ode_func(D_tensor).numpy()

print('─── Verificación Neural ODE ───────────────────────────')
for label, D_mid, dD, lam_key in [
    ('hot',  D_mid_h, dD_pred[0], 'hot'),
    ('warm', D_mid_w, dD_pred[1], 'warm')]:
    lam = lambdas.get(lam_key, float('nan'))
    slope_expected = -lam * D_mid
    print(f'[{label:4s}]  ODE f(D_mid) = {dD:.5f}   '
          f'esperado (-λ₂·D) = {slope_expected:.5f}')


## 7. Resumen y próximos pasos

| Aspecto | Neural ODE (RK4) | PINN | PySR |
|---------|-----------------|------|------|
| Integrador | RK4 ($O(\\Delta t^5)$) | n/a (mapeo $t\\to D$) | n/a |
| Restricción física | ninguna explícita | $dD/dt \\le 0$ via autograd | ninguna |
| Interpretabilidad | media ($f_\\theta$ caja negra) | baja | **alta** (ecuación simbólica) |
| Detecta $t^*$ | ✓ si ajusta bien | ✓ si ajusta bien | indirectamente |

**Próximos pasos:**

1. Si PySR confirma $\\dot{D} \\approx -k D$, reemplaza la restricción en la PINN
   por el residuo `dD/dt + k*D = 0` (ver comentario en Sección 3).
2. Extiende el análisis a `D_KL_h` / `D_KL_w` para verificar que el cruce
   es independiente de la métrica — Extended Data Fig. 7 del paper.
3. Compara $\\lambda_2$ extraído aquí con el eigenvalor analítico de la ecuación
   de Fokker-Planck para el potencial `colloid_kumar.ini`.
4. Para extrapolación fuera del rango de entrenamiento, reemplaza el
   integrador manual por `torchdiffeq` (método adjunto).
